In [5]:
import os
import re
import ast
import time
import pandas as pd
import requests
from dotenv import load_dotenv

# Load API keys from root .env
load_dotenv(dotenv_path="../../../.env")
if not os.getenv("OPENROUTER_API_KEY"):
    load_dotenv()

# Store all available keys in a list
api_keys = []
key1 = os.getenv("OPENROUTER_API_KEY")
key2 = os.getenv("OPENROUTER_API_KEY_NEW")

if key1: api_keys.append(key1)
if key2: api_keys.append(key2)

if not api_keys:
    raise ValueError("No API keys found! Please verify your .env file.")

# Set evaluation model to Llama 3.1 70B
EVAL_MODEL = "qwen/qwen-2.5-72b-instruct"

print(f"Environment ready. Loaded {len(api_keys)} API keys.")
print(f"Evaluation Model set to: {EVAL_MODEL}")

Environment ready. Loaded 2 API keys.
Evaluation Model set to: qwen/qwen-2.5-72b-instruct


In [7]:
# The FULL dataset generated by GPT-5.6 Luna
INPUT_FILE = "gpt5.6_luna_health_condition_full_dataset.csv"
OUTPUT_FILE = "gpt5.6_luna_full_evaluated_by_qwen_2_5_72b.csv"
BACKUP_FILE = "backup_gpt5.6_luna_full_evaluated_by_qwen_2_5_72b.csv"

print(f"Loading full generated dataset from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

def extract_target_condition(row):
    """Extracts target health condition from available columns."""
    if "target_health_condition" in row and pd.notna(row["target_health_condition"]):
        return str(row["target_health_condition"]).strip()

    val = row.get("Health Condition")
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if val_str.startswith("[") and val_str.endswith("]"):
        try:
            parsed = ast.literal_eval(val_str)
            if isinstance(parsed, list) and len(parsed) > 0:
                return str(parsed[0]).strip()
        except Exception:
            pass
    return val_str

df["eval_target_condition"] = df.apply(extract_target_condition, axis=1)
df = df.dropna(subset=["text", "modified_sentence", "eval_target_condition"]).reset_index(drop=True)
print(f"Loaded {len(df)} rows from full dataset.")

# ---- RESUME: pull in whatever the previous run already completed ----
# We key on "text" (unique per row) and copy over any non-null predictions
# from the backup so we only re-hit the API for what actually failed.
for col in ["correct_prediction", "correct_raw_response",
            "hallucinated_prediction", "hallucinated_raw_response"]:
    if col not in df.columns:
        df[col] = None

try:
    bk = pd.read_csv(BACKUP_FILE)
    bk_indexed = bk.set_index("text")

    resumed = 0
    for i, row in df.iterrows():
        key = row["text"]
        if key in bk_indexed.index:
            b = bk_indexed.loc[key]
            # if duplicate texts exist in backup, take the first
            if isinstance(b, pd.DataFrame):
                b = b.iloc[0]
            for col in ["correct_prediction", "correct_raw_response",
                        "hallucinated_prediction", "hallucinated_raw_response"]:
                if pd.notna(b.get(col)):
                    df.at[i, col] = b[col]
            resumed += 1

    # Normalize prediction columns to nullable ints (0/1/NaN)
    for col in ["correct_prediction", "hallucinated_prediction"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    print(f"Resumed {resumed} rows from backup ({BACKUP_FILE}).")
except FileNotFoundError:
    print(f"No backup found at {BACKUP_FILE} - starting fresh.")

n_orig_left = df["correct_prediction"].isna().sum()
n_hall_left = df["hallucinated_prediction"].isna().sum()
print(f"Still to do -> Original: {n_orig_left} | Hallucinated: {n_hall_left} "
      f"| API calls: {int(n_orig_left + n_hall_left)}")


Loading full generated dataset from gpt5.6_luna_health_condition_full_dataset.csv...
Loaded 577 rows from full dataset.
Resumed 550 rows from backup (backup_gpt5.6_luna_full_evaluated_by_qwen_2_5_72b.csv).
Still to do -> Original: 157 | Hallucinated: 154 | API calls: 311


In [8]:
PROMPT_TEMPLATE = """
You are a medical verification assistant.

Determine whether the extracted medical entity is hallucinated with respect to the given text.

Text:
{text}

Extracted Entity:
{target_entity}

If the extracted entity is hallucinated, output:
ANSWER: 1

Otherwise, output:
ANSWER: 0
"""

def evaluate_entity(text, target_entity, model_name=EVAL_MODEL, max_retries=3):
    """Calls OpenRouter chat/completions with retry+backoff.

    Retries handle two known failure modes seen with qwen-2.5-72b-instruct
    on OpenRouter (which only has 2 upstream providers behind it):
      1. A transient 400 from one provider ("does not support endpoint: completions")
         - retrying often lands on the healthy provider instead.
      2. Rate limiting (429) or a brief outage window - backoff gives it time to clear.
    """
    prompt = PROMPT_TEMPLATE.format(
        text=str(text).strip(),
        target_entity=str(target_entity).strip()
    )

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0,
        # Nudge OpenRouter away from a currently-stuck provider rather than
        # always hammering the same (possibly unhealthy) one.
        "provider": {"sort": "throughput"}
    }

    last_err = None
    for attempt in range(max_retries):
        try:
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=120
            )

            # Read the body BEFORE raising on status, so we can see the
            # real OpenRouter error message instead of a generic 400.
            try:
                result = response.json()
            except ValueError:
                response.raise_for_status()
                raise ValueError(f"Non-JSON response (status {response.status_code}): {response.text[:300]}")

            if "choices" not in result:
                err_msg = result.get("error", {}).get("message", result)
                raise ValueError(f"OpenRouter did not return choices: {err_msg}")

            content = result["choices"][0]["message"]["content"].strip()
            match = re.search(r"ANSWER:\s*([01])", content, re.IGNORECASE)
            prediction = int(match.group(1)) if match else None

            return prediction, content

        except Exception as e:
            last_err = e
            if attempt < max_retries - 1:
                wait = 3 * (attempt + 1)  # 3s, 6s, ...
                time.sleep(wait)

    # All retries exhausted - bubble up the last error to the caller.
    raise last_err


In [9]:
# Build lists seeded from whatever we already have, so partial progress is preserved.
correct_predictions = df["correct_prediction"].tolist()
hall_predictions = df["hallucinated_prediction"].tolist()
correct_raw = df["correct_raw_response"].tolist()
hall_raw = df["hallucinated_raw_response"].tolist()

def _isna(x):
    return x is None or (isinstance(x, float) and pd.isna(x))

to_do = [i for i in range(len(df)) if _isna(correct_predictions[i]) or _isna(hall_predictions[i])]
print(f"Evaluating {len(to_do)} rows that are missing a prediction "
      f"(of {len(df)} total) using {EVAL_MODEL}...\n")

processed = 0
for i in to_do:
    row = df.loc[i]
    orig_text = row["text"]
    mod_text = row["modified_sentence"]
    target_condition = row["eval_target_condition"]

    # Only call the API for the side that's actually missing.
    if _isna(correct_predictions[i]):
        try:
            pred_orig, raw_orig = evaluate_entity(orig_text, target_condition)
        except Exception as e:
            print(f"Row {i+1} [Original] Error: {e}")
            pred_orig, raw_orig = None, str(e)
        correct_predictions[i] = pred_orig
        correct_raw[i] = raw_orig

    if _isna(hall_predictions[i]):
        try:
            pred_hall, raw_hall = evaluate_entity(mod_text, target_condition)
        except Exception as e:
            print(f"Row {i+1} [Hallucinated] Error: {e}")
            pred_hall, raw_hall = None, str(e)
        hall_predictions[i] = pred_hall
        hall_raw[i] = raw_hall

    print(f"[{i+1}/{len(df)}] Orig Pred: {correct_predictions[i]} (Exp: 0) "
          f"| Hall Pred: {hall_predictions[i]} (Exp: 1)")

    processed += 1
    # Auto-save every 50 processed rows against the full df.
    if processed % 50 == 0:
        df["correct_prediction"] = correct_predictions
        df["correct_raw_response"] = correct_raw
        df["hallucinated_prediction"] = hall_predictions
        df["hallucinated_raw_response"] = hall_raw
        df.to_csv("backup_" + OUTPUT_FILE, index=False, encoding="utf-8-sig")
        print(f"   --- Auto-saved backup after {processed} processed rows ---")

    time.sleep(0.5)

# Save first-pass results
df["correct_prediction"] = correct_predictions
df["correct_raw_response"] = correct_raw
df["hallucinated_prediction"] = hall_predictions
df["hallucinated_raw_response"] = hall_raw
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\nFirst pass complete. Saved to {OUTPUT_FILE}")


Evaluating 170 rows that are missing a prediction (of 577 total) using qwen/qwen-2.5-72b-instruct...

[6/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1.0 (Exp: 1)
[55/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1.0 (Exp: 1)
[56/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[57/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[58/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1.0 (Exp: 1)
[59/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[60/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 0.0 (Exp: 1)
[61/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1.0 (Exp: 1)
[62/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[63/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1.0 (Exp: 1)
[64/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1.0 (Exp: 1)
[66/577] Orig Pred: 0.0 (Exp: 0) | Hall Pred: 1 (Exp: 1)
[107/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1.0 (Exp: 1)
[108/577] Orig Pred: 0.0 (Exp: 0) | Hall Pred: 0 (Exp: 1)
[109/577] Orig Pred: 0 (Exp: 0) | Hall Pred: 1.0 (Exp: 1)
[110/577] Orig Pred: 0.0 (Exp: 0) | Hall Pred: 1 

In [11]:
# ---- RETRY PASS: re-attempt any row still missing a prediction ----
# Outages on OpenRouter (2 providers behind this model) come in windows; a pass
# after the first sweep finishes recovers most of what inline retries couldn't.

def _isna(x):
    return x is None or (isinstance(x, float) and pd.isna(x))

retry_idx = [i for i in range(len(df))
             if _isna(correct_predictions[i]) or _isna(hall_predictions[i])]
print(f"{len(retry_idx)} rows still need retrying.")

for i in retry_idx:
    row = df.loc[i]
    orig_text = row["text"]
    mod_text = row["modified_sentence"]
    target_condition = row["eval_target_condition"]

    if _isna(correct_predictions[i]):
        try:
            pred_orig, raw_orig = evaluate_entity(orig_text, target_condition)
            correct_predictions[i] = pred_orig
            correct_raw[i] = raw_orig
        except Exception as e:
            print(f"Retry row {i+1} [Original] still failing: {e}")

    if _isna(hall_predictions[i]):
        try:
            pred_hall, raw_hall = evaluate_entity(mod_text, target_condition)
            hall_predictions[i] = pred_hall
            hall_raw[i] = raw_hall
        except Exception as e:
            print(f"Retry row {i+1} [Hallucinated] still failing: {e}")

    print(f"[retry {i+1}/{len(df)}] Orig Pred: {correct_predictions[i]} (Exp: 0) "
          f"| Hall Pred: {hall_predictions[i]} (Exp: 1)")
    time.sleep(0.5)

# ---- Final save + metrics ----
df["correct_prediction"] = correct_predictions
df["correct_raw_response"] = correct_raw
df["hallucinated_prediction"] = hall_predictions
df["hallucinated_raw_response"] = hall_raw
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

total_rows = len(df)
correct_matches = int((df["correct_prediction"] == 0).sum())
hall_matches = int((df["hallucinated_prediction"] == 1).sum())
still_failed = int(df["correct_prediction"].isna().sum() + df["hallucinated_prediction"].isna().sum())

correct_accuracy = (correct_matches / total_rows) * 100 if total_rows > 0 else 0
hall_accuracy = (hall_matches / total_rows) * 100 if total_rows > 0 else 0

print("\n==============================")
print(f"Evaluator Model                : {EVAL_MODEL}")
print(f"Correct Sentence Accuracy      : {correct_accuracy:.2f}%")
print(f"Hallucinated Sentence Accuracy : {hall_accuracy:.2f}%")
print(f"Predictions still unresolved   : {still_failed}")
print(f"Saved detailed results to      : {OUTPUT_FILE}")
print("==============================")


0 rows still need retrying.

Evaluator Model                : qwen/qwen-2.5-72b-instruct
Correct Sentence Accuracy      : 98.09%
Hallucinated Sentence Accuracy : 67.94%
Predictions still unresolved   : 0
Saved detailed results to      : gpt5.6_luna_full_evaluated_by_qwen_2_5_72b.csv
